# Trace recurrent computation and masked objectives

Use scalar fixtures to expose recurrent state, shared derivatives, gate updates and token loss reduction.

Fill in each starter, then run its check cell. Restart the kernel and run all cells before submitting. Keep your interpretation in the last Markdown cell.


## 1. Implement unroll a recurrent state

Unroll a tanh recurrence.

Implement unroll(inputs,w,u,b,initial) using the stated tanh recurrence. Return each new hidden state in order, excluding the initial state.


In [ ]:
def unroll(inputs,w,u,b,initial):
    raise NotImplementedError


In [ ]:
from math import tanh, isclose
assert unroll([],2,1,0,3)==[]
assert unroll([1,0],0.5,1,0,0)==[tanh(1),tanh(0.5*tanh(1))]
assert unroll([0,0],0,1,0,1)==[0,0]
assert unroll([0],2,0,0,1)==[tanh(2)]
r = unroll([1,0],0.5,1,0.25,0)
assert len(r)==2 and all(isclose(a,e) for a,e in zip(r,[tanh(1.25),tanh(0.5*tanh(1.25)+0.25)])), "add b inside tanh at every step"
print("OK")


## 2. Implement backpropagate through time

Differentiate a linear sequence end to end.

Implement linear_bptt(inputs,w,initial) for h_t=w*h_previous+x_t and loss equal to final state. Return (value, dvalue/dw). Treat initial as fixed.


In [ ]:
def linear_bptt(inputs,w,initial):
    raise NotImplementedError


In [ ]:
assert linear_bptt([1,2,3],2,0)==(11,6)
assert linear_bptt([0],2,3)==(6,3)
assert linear_bptt([1,2],0,0)==(2,1)
assert linear_bptt([],2,3)==(3,0)
print("OK")


## 3. Implement lstm gated state

Compute an LSTM cell from activated gates.

Implement lstm_from_gates. Inputs are an old cell value, activated forget, input and output gates within [0,1], and an activated candidate in [-1,1]. Return (new cell, hidden output).


In [ ]:
def lstm_from_gates(cell,forget,insert,candidate,expose):
    raise NotImplementedError


In [ ]:
from math import tanh
assert lstm_from_gates(2,0.5,1,0.5,0)==(1.5,0)
assert lstm_from_gates(3,1,0,-1,1)==(3,tanh(3))
assert lstm_from_gates(3,0,1,-1,0.5)==(-1,0.5*tanh(-1))
assert lstm_from_gates(1,0,0,1,1)==(0,0)
print("OK")


## 4. Implement mask padded token loss

Exclude padding from a token average.

Implement masked_token_mean(losses,mask). Return the mean over true mask positions, None when none are included, and ValueError when lengths differ.


In [ ]:
def masked_token_mean(losses,mask):
    raise NotImplementedError


In [ ]:
assert masked_token_mean([1,5,99],[True,True,False])==3
assert masked_token_mean([0,4],[True,False])==0
assert masked_token_mean([1],[False]) is None
assert masked_token_mean([],[]) is None
try:
    masked_token_mean([1],[])
except ValueError:
    pass
else:
    raise AssertionError("reject mismatched mask")
print("OK")


## 5. Differentiate the nonlinear recurrence and inspect truncation

Implement tanh_weight_gradient(inputs,w,u,b,initial,steps=None) for final hidden-state loss.
Reuse unroll. Walk backward from the final state for at most steps transitions, with None
meaning the full sequence. Treat the boundary state as detached when truncating. For each reverse
step, multiply the state adjoint by 1-state**2 to get the preactivation adjoint, add that times
the previous state to the shared w gradient, and propagate through w. Return (final state,
weight gradient). A cap of zero gives a zero gradient. Inputs are finite, and steps is nonnegative
or None. Compare the full derivative with centered differences.


In [ ]:
def tanh_weight_gradient(inputs,w,u,b,initial,steps=None):
    raise NotImplementedError


In [ ]:
inputs,w,u,b,initial = [0.3,-0.4,0.7],0.65,0.8,0.1,0.25
value,gradient = tanh_weight_gradient(inputs,w,u,b,initial)
h = 1e-5
numeric = (unroll(inputs,w+h,u,b,initial)[-1]-unroll(inputs,w-h,u,b,initial)[-1])/(2*h)
assert abs(gradient-numeric)<1e-8
assert abs(gradient-0.2895672201468395)<1e-9
truncated = tanh_weight_gradient(inputs,w,u,b,initial,1)
assert truncated[0]==value and abs(truncated[1]-gradient)>1e-5
assert abs(truncated[1]-0.050904321631530206)<1e-9
assert tanh_weight_gradient(inputs,w,u,b,initial,0)==(value,0)
assert tanh_weight_gradient([],w,u,b,initial)==(initial,0)
print("OK")


## Interpretation

Explain what changes if you detach the state halfway through a long sequence. Give an example where masking the loss alone cannot repair the forward path.

**Your answer**
